# GFM Coasts — Notebook 1: Setting the Stage

**Applying Geographic Foundation Models to coastal erosion along the Catalan coast.**

## What we're trying to do

The Catalan coast is changing. The Ebro Delta is one of the fastest-retreating deltas in Europe — losing ground at metres per year in some places, starved of sediment by the upstream dams (Mequinenza, Riba-roja). Sixty kilometres up the coast, the Maresme has chronic sand-deficit pocket beaches that get nourished, eroded, nourished again. And every few years, a Mediterranean storm — like Gloria in January 2020 — rewrites all of it overnight.

This project is **exploratory**. It asks two layered questions:

1. **The science question:** *How fast and where is the Catalan coast eroding, and how does it respond to extreme storm events?*
2. **The methods question:** *What can a Geographic Foundation Model (GFM) reveal about this coast — and how do its outputs sit alongside what established methods (CoastSat shorelines, DSAS change rates, LiDAR DEM differencing) typically produce?*

We are not running a head-to-head benchmark and we are not trying to declare a winner. We will run the GFM pipeline honestly, produce numerical outputs (shoreline positions, change rates, change masks), and place those numbers next to *published reference values* from CoastSat-and-similar studies for the same or analogous coast — so a reader can see where our outputs land. Whether that's reassuring, surprising, or inconclusive is part of what we want to find out.


## Where we're looking

Two areas with very different physics:

### Ebro Delta
A sediment-starved delta. The Ebro carries far less sand to the coast than it used to (dams trap most of it), and the wave climate redistributes the existing delta. The northern hemidelta (Punta del Fangar) is gaining; the southern (Punta de la Banya) is losing dramatically. Some studies put retreat rates in the high single digits of metres per year.

### Maresme (Barcelona–Tordera)
The coast immediately north of Barcelona. Pocket beaches squeezed between rocky headlands and a railway line at the back of the beach (so even small erosion causes infrastructure damage). Heavily nourished, but the underlying deficit is structural. UPC-CIIRC researchers (Sánchez-Arcilla, Jiménez and colleagues) have studied this stretch for decades.

These two AOIs are interesting precisely because they fail differently. We don't pre-commit to expecting the GFM to behave the same way on both — that's part of what we'll find out.


## What is a Geographic Foundation Model?

> **Analogy:** Think of a foundation model as a *polyglot interpreter*. They've spent years listening to thousands of conversations in dozens of languages — they don't have a job yet, but they've internalised grammar, idiom, accent. When you give them a specific job (translate a contract, transcribe a meeting), they learn it fast, with very little task-specific training, because they already understand how language works.
>
> A traditional CNN trained from scratch is a *tourist with a phrasebook for one task*. It can recognise "where is the bathroom" perfectly, but knows nothing else.

A **Geographic Foundation Model** (GFM) is a foundation model whose pre-training was specifically on Earth-observation data: Sentinel-2 multispectral imagery, sometimes Landsat, sometimes SAR, sometimes LiDAR or HLS. The pre-training objective is usually self-supervised — masked autoencoding (MAE), contrastive learning, or some flavour of both — meaning the model learns to *predict missing image patches* across millions of unlabelled satellite tiles.

What this internalises:

- **Spectral grammar.** Vegetation indices, water signatures, soil response — the model learns these without ever being told what NDVI is.
- **Spatial context.** What a coastline tends to look like. What a city looks like. What a flood looks like.
- **Temporal structure** (in some models). Seasonal cycles, gradual change vs sudden change.
- **Scale invariance** (in some models). The same scene at 10 m vs 30 m resolution.

You then **fine-tune** the pre-trained model for your specific task — segmentation, classification, change detection, regression — or just use its **embeddings** directly as features. Either way, you typically need *far fewer labels* than training from scratch.

### Where GFMs shine
- Tasks with limited labels (most of EO!)
- Multi-modal fusion (S2 + SAR + DEM)
- Change detection over long time series
- Transfer to new geographies

### Where GFMs may not help
- Tasks already solved well by classical methods with abundant labels
- Tasks dominated by physics (radiative transfer, wave equations) rather than pattern recognition
- Tasks needing sub-pixel accuracy that the pre-training resolution doesn't support


## The GFM landscape (May 2026)

A non-exhaustive map of the models we'll consider, in rough order of "use first":

| Model | Why pick it | Trade-off |
|---|---|---|
| **Clay v1.5** | Open, embeddings-first, S2-native, runs on CPU for small AOIs | Embeddings, not segmentation maps — you build the downstream task |
| **Prithvi-EO-2.0** (IBM/NASA) | Fine-tuning recipes for segmentation and multi-temporal change detection; HLS-pretrained | Fine-tuning needs a GPU; bigger weights |
| **TerraMind** (IBM) | Multimodal — handles SAR alongside optical | Heavier; more complex to set up |
| **DOFA** | Dynamic input bands (handles missing/extra channels gracefully) | Less coastal-specific work in the literature |
| **SatMAE / ScaleMAE** | Earlier MAE-based GFMs | Older; superseded for most tasks |

### Our plan
1. **Start with Clay v1.5 embeddings.** Cheap, lets us answer "what does the GFM see in this coast?" before investing in a fine-tune.
2. **Move to Prithvi-EO-2.0 fine-tune** if Clay reveals enough structure to be worth the next step — for explicit water/sand/vegetation segmentation and per-pixel change masks.
3. **Reach for TerraMind** for the Storm Gloria zoom, where SAR is valuable (clouds during the storm; rapid revisit).

### Why embeddings before fine-tuning?
A foundation-model embedding for a satellite tile is a fixed-length vector that compresses everything the model "thinks" about that tile. You can:

- Cluster embeddings to discover coastal types automatically
- Compute cosine distance between embeddings of the same place at different times — that is a *change signal* with no labels needed
- Use embeddings as features for a tiny downstream classifier

If embedding-based change detection visibly tracks the erosion patterns we already know exist (Ebro south retreat, Maresme winter losses), that's a meaningful signal. If it doesn't, fine-tuning may not save it — but we'd have learned something either way.


## Why coastal erosion is a good GFM testcase

Coastal erosion sits in a sweet spot for exploring GFMs:

1. **Multi-modal by nature.** Optical imagery (S2), SAR (Sentinel-1, clouds don't matter), DEMs (LiDAR), bathymetry — a model that fuses these has access to richer signal than a single-source classifier.
2. **Multi-temporal.** Change is the signal; static snapshots are not enough. GFMs with temporal pre-training (Prithvi, some Clay variants) can take time as a first-class input.
3. **Hard to label.** Manual shoreline digitisation is slow and expensive. A method that works from few labels (or none, via embeddings) is genuinely useful.
4. **Strong physical priors exist.** Wave climate, sediment supply, tidal range — we can sanity-check whether the model's outputs respect physics. If the GFM tells us the southern Ebro hemidelta is *gaining* sediment, something is wrong.

### A creative provocation

> Classical shoreline detection draws a *line* between water and land. That's a 1-D answer to a 3-D question. What if the more useful output isn't a line at all, but a *map of beach state* — narrow/wide, dry/wet, vegetated/unvegetated, scarp/ramp — that a GFM can produce because it has seen millions of beaches?

Worth holding on to as a hypothesis when we get to chapter 4.


## Workflow at a glance

```mermaid
flowchart LR
    A[ICGC LiDARCAT<br/>+ orthophotos] --> V[Validation<br/>shorelines]
    B[Sentinel-2 L2A<br/>2017-2025] --> C[CoastSat<br/>baseline shoreline]
    B --> D[Clay v1.5<br/>embeddings]
    B --> E[Prithvi-EO-2.0<br/>fine-tune segmentation]
    F[Sentinel-1 SAR] --> E
    F --> G[TerraMind<br/>storm zoom]
    C --> H[DSAS-style<br/>change rates]
    D --> I[Embedding-distance<br/>change detection]
    E --> J[Per-pixel<br/>change masks]
    V --> K[Side-by-side<br/>with literature]
    H --> K
    I --> K
    J --> K
    L[Wave climate<br/>Puertos del Estado] --> M[Synthesis<br/>+ maps]
    N[Ebro discharge<br/>CHE] --> M
    K --> M
```

Each downstream notebook corresponds to one or two boxes:

- **02:** Data acquisition (everything above the diagonal)
- **03:** CoastSat + DSAS-style baseline (boxes C, H)
- **04:** Clay embeddings (D, I)
- **05:** Prithvi fine-tune (E, J) — *only if 04 justifies it*
- **06:** Side-by-side with LiDAR truth and literature reference values (V, K)
- **07:** Storm Gloria zoom (F, G)
- **08:** Synthesis and interactive maps (M)


## Choices we've made (and want to be honest about)

| Choice | Why | Alternative |
|---|---|---|
| Clay first, Prithvi later | Cheap to evaluate "is the GFM seeing anything?" | Skip embeddings, go straight to fine-tuned Prithvi |
| LiDAR-derived shoreline as truth | Most accurate ground truth available; ICGC has multiple campaigns | Manual orthophoto digitisation (more work, cleaner) |
| CoastSat applied alongside, not as a competitor | We want a familiar set of numbers next to GFM outputs, not a winner | Skip the classical pipeline entirely (less context for the reader) |
| Reference RMSEs from published CoastSat studies | We're not claiming our GFM beats anything — we're showing where our numbers land | Pre-register a head-to-head comparison (premature; we're still learning) |
| Ebro Delta + Maresme | Two failure modes (delta retreat vs pocket-beach deficit) | Single AOI (cheaper) or whole coast (richer but heavier) |
| Decadal trend + Gloria zoom | Long signal + extreme event; pedagogically rich | Trend only or storm only |

## Open questions to keep in mind

- **What is the right tidal datum for shoreline derivation from LiDAR?** Mediterranean tides are tiny (~20–30 cm range), so the sensitivity is low — but storm surge during Gloria reached over a metre. We'll need to be careful which datum we use for which comparison.
- **How much do we trust pre-2017 data?** Sentinel-2 starts 2015 (S2A) / 2017 (S2B). For longer trends we'd need Landsat, but Landsat is 30 m — coarse for shoreline work.
- **Should we use Sentinel-1 SAR for the trend, not just Gloria?** SAR ignores clouds. Coastal Mediterranean is sometimes cloudy in winter. But SAR shorelines are noisy.
- **Embeddings give us a similarity score, not a shoreline.** What's a sensible way to *project* embedding-distance into a unit a coastal scientist would recognise (metres of retreat, m³ of sediment)?
- **What does "the result" look like?** A map of erosion rates with uncertainty? A side-by-side gallery of GFM-segmented coast next to CoastSat lines? A short report? Default for now: all three, modest in scope.

These are good questions to revisit at the end of each notebook.


In [1]:
# Minimal environment check.
# Heavier imports start in notebook 02.
import sys, platform, pathlib

PROJECT_ROOT = pathlib.Path('..').resolve()
print(f'Python:        {sys.version.split()[0]}')
print(f'Platform:      {platform.system()} {platform.release()}')
print(f'Project root:  {PROJECT_ROOT}')
print()
print('Folder layout:')
for p in sorted(PROJECT_ROOT.iterdir()):
    if not p.name.startswith('.'):
        print(f'  {p.name}/' if p.is_dir() else f'  {p.name}')


Python:        3.11.13
Platform:      Darwin 25.4.0
Project root:  /Users/jamesmurphy/Documents/Documents/GitHub/GFM_coasts

Folder layout:
  README.md
  data/
  maps/
  notebooks/
  pyproject.toml
  reports/
  src/
  uv.lock


## What's next

**Notebook 02 — Data acquisition.** We'll define the two AOI polygons (Ebro Delta + Maresme), pull a Sentinel-2 L2A time series via STAC for both, fetch ICGC LiDAR tiles and orthophotos, and grab wave records from Puertos del Estado plus Ebro discharge from CHE.

Before opening notebook 02:

1. **Install the environment.** This project uses [uv](https://docs.astral.sh/uv/). From the project root: `uv sync`. To pull the heavier GFM frameworks too: `uv sync --extra gfm`.
2. **Sanity-check this notebook.** Does the framing match what you wanted? Are there choices in the table above you want to renegotiate? Now is the cheapest moment to change direction.
